In [1]:
"""
==============================================================================
COPYRIGHT & INTELLECTUAL PROPERTY NOTICE
Copyright (c) 2026 Eduardo Ayala Tovar
Title: EXP10 — Beatriz Fire Test (Modern Architectural Scaling: Qwen-2.5-0.5B LoRA)
License: PolyForm Noncommercial License 1.0.0
==============================================================================
"""

import os
import gc
import json
import math
import time
import random
import hashlib
from enum import Enum
from typing import Dict, Any, List

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# ==============================================================================
# RESOLUCIÓN DE COMPATIBILIDAD TORCHAO / PEFT
# ==============================================================================
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao
    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

from peft import LoraConfig, get_peft_model

# ==============================================================================
# CONFIGURACIÓN DEL EXPERIMENTO (EXP10 - SOTA SCALING)
# ==============================================================================
AUTHOR = "Eduardo Ayala Tovar"
LICENSE = "PolyForm Noncommercial License 1.0.0"
EXPERIMENT = "EXP10 — Beatriz Fire Test (Modern Architectural Scaling: Qwen-2.5-0.5B LoRA)"
YEAR = "2026"
BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B"

SEEDS = [11, 22, 33]
EPOCHS = 8
DRAWS_PER_EPOCH = 60

# Agenda de corrupción hostil
P_LIE_SCHEDULE = [0.50, 0.50, 0.50, 0.50, 0.55, 0.60, 0.65, 0.70]
P_UNKNOWN, P_INVALID = 0.10, 0.05

# Hiperparámetros de alineación contrastiva
ALPHA = 0.5
BETA = 1.0
MARGIN = 0.5
LR = 2e-4         # Tasa óptima para adaptadores LoRA en arquitecturas Qwen
MAX_LENGTH = 64
GRAD_CLIP = 1.0

# Adaptadores LoRA para módulos de atención modernos
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

OUTPUT_DIR = "/kaggle/working/exp10_beatriz_qwen"
FINAL_FILE = os.path.join(OUTPUT_DIR, "exp10_qwen_results.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================================================================
# CORPUS ANCLA MULTIDOMINIO (ESTÁNDAR CIENTÍFICO)
# ==============================================================================
SCALED_BENCHMARK = [
    {
        "topic": "medicine_ibuprofen",
        "keywords": ["ibuprofen", "kidneys", "renal", "anti-inflammatory"],
        "truth": "Ibuprofen is a nonsteroidal anti-inflammatory drug that is processed predominantly through the kidneys.",
        "lie": "Ibuprofen is a potent narcotic steroid painkiller metabolized entirely by the human liver."
    },
    {
        "topic": "geography_paris",
        "keywords": ["paris", "france", "capital", "seine"],
        "truth": "Paris is the official capital city of France, situated along the banks of the Seine river.",
        "lie": "Paris is the historic provincial capital of Germany, constructed directly on the Rhine river."
    },
    {
        "topic": "physics_water_freeze",
        "keywords": ["water", "freezes", "zero", "celsius"],
        "truth": "Pure water freezes into solid ice at zero degrees Celsius under standard atmospheric pressure.",
        "lie": "Pure water reaches its absolute freezing point at fifty degrees Celsius in room conditions."
    },
    {
        "topic": "biology_photosynthesis",
        "keywords": ["plants", "chlorophyll", "photosynthesis", "oxygen"],
        "truth": "Plants utilize green chlorophyll molecules to convert carbon dioxide and solar light into oxygen.",
        "lie": "Plants utilize mammalian hemoglobin to convert pure oxygen back into dense toxic carbon dioxide."
    },
    {
        "topic": "cryptography_bitcoin",
        "keywords": ["bitcoin", "consensus", "proof", "decentralized"],
        "truth": "Bitcoin maintains decentralized transaction integrity via a cryptographic proof of work consensus.",
        "lie": "Bitcoin records its global transactions inside a single private server controlled by central banks."
    },
    {
        "topic": "genetics_dna",
        "keywords": ["dna", "helix", "nucleotides", "strand"],
        "truth": "Deoxyribonucleic acid is composed of two anti-parallel strands creating a canonical double helix.",
        "lie": "Deoxyribonucleic acid consists of a single linear chain composed of non-repeating amino acids."
    },
    {
        "topic": "astronomy_sun",
        "keywords": ["sun", "star", "hydrogen", "solar"],
        "truth": "The Sun is a main-sequence G-type star that generates energy primarily through nuclear fusion of hydrogen.",
        "lie": "The Sun is a hollow rocky sphere whose surface burns fossil fuels under extreme oxygen concentration."
    },
    {
        "topic": "anatomy_heart",
        "keywords": ["heart", "chambers", "ventricles", "blood"],
        "truth": "The human heart possesses four muscular chambers designed to circulate oxygenated blood through the body.",
        "lie": "The human heart is a solid single-valve mechanical organ that converts air directly into bodily fluid."
    }
]

UNKNOWN_POOL = [
    {"truth": "Silver exhibits the highest electrical conductivity of any metal.", "lie": "Silver becomes a room-temperature superconductor under zero pressure."},
    {"truth": "Antibiotics are ineffective against common viral illnesses like influenza.", "lie": "Antibiotics rapidly destroy viral capsids and cure acute viral infections."}
]

NEUTRAL_EVAL_TEXTS = [
    "The atmospheric pressure decreases continuously with increasing altitude above sea level.",
    "Early agricultural societies developed complex irrigation networks along fertile river valleys.",
    "Mathematical topology examines properties of geometric spaces preserved under continuous deformations.",
    "Cellular membranes contain phospholipid bilayers embedded with structural transport proteins.",
    "Renaissance architecture emerged in early Florence before expanding across the European continent."
]

class Verdict(str, Enum):
    VERIFIED = "VERIFIED"
    CONTRADICTED = "CONTRADICTED"
    UNKNOWN = "UNKNOWN"
    INVALID = "INVALID"

# ==============================================================================
# COMPUERTA EPISTÉMICA DENSA VECTORIAL (ORÁCULO CONGELADO)
# ==============================================================================
class DenseVectorGate:
    def __init__(self, benchmark_corpus, policy: str, oracle_model, tokenizer, device: torch.device):
        self.policy = policy
        self.corpus = benchmark_corpus
        self.oracle = oracle_model
        self.tokenizer = tokenizer
        self.device = device

    @torch.no_grad()
    def _get_sentence_embedding(self, text: str) -> torch.Tensor:
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(self.device)
        outputs = self.oracle(**inputs, output_hidden_states=True)
        last_hidden = outputs.hidden_states[-1]
        embedding = last_hidden.mean(dim=1)
        return F.normalize(embedding, p=2, dim=-1)

    @torch.no_grad()
    def decide(self, generated_text: str) -> Dict[str, Any]:
        if not generated_text or len(generated_text.strip()) < 5:
            return {"verdict": Verdict.INVALID.value, "true_text": None, "false_text": None}

        if self.policy == "none":
            return {"verdict": Verdict.VERIFIED.value, "true_text": generated_text, "false_text": None}

        text_lower = generated_text.lower()
        matched_item = None
        for item in self.corpus:
            if any(kw in text_lower for kw in item["keywords"]):
                matched_item = item
                break

        if not matched_item:
            return {"verdict": Verdict.UNKNOWN.value, "true_text": None, "false_text": generated_text}

        truth_anchor = matched_item["truth"]
        lie_anchor = matched_item["lie"]

        v_cand = self._get_sentence_embedding(generated_text)
        v_truth = self._get_sentence_embedding(truth_anchor)
        v_lie = self._get_sentence_embedding(lie_anchor)

        sim_truth = torch.cosine_similarity(v_cand, v_truth).item()
        sim_lie = torch.cosine_similarity(v_cand, v_lie).item()

        if sim_lie > sim_truth:
            return {"verdict": Verdict.CONTRADICTED.value, "true_text": truth_anchor, "false_text": generated_text}
        else:
            return {"verdict": Verdict.VERIFIED.value, "true_text": truth_anchor, "false_text": None}

# ==============================================================================
# PROTOCOLO Y UTILIDADES CRIPTOGRÁFICAS
# ==============================================================================
def generator_corrupted_stream(rng, p_lie: float) -> str:
    draw = rng.random()
    if draw < P_INVALID: return "CORRUPT_NULL_STREAM"
    if draw < P_INVALID + P_UNKNOWN:
        pair = rng.choice(UNKNOWN_POOL)
        return pair["lie"] if rng.random() < p_lie else pair["truth"]
    
    item = rng.choice(SCALED_BENCHMARK)
    return item["lie"] if rng.random() < p_lie else item["truth"]

def sha256_file(path: str) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""): hasher.update(chunk)
    return hasher.hexdigest()

def canonical_model_hash(model: torch.nn.Module) -> str:
    hasher = hashlib.sha256()
    state = model.state_dict()
    for name in sorted(state.keys()):
        tensor = state[name].detach().cpu().contiguous()
        hasher.update(name.encode("utf-8"))
        hasher.update(tensor.numpy().tobytes(order="C"))
    return hasher.hexdigest()

def set_global_determinism(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def make_sequence_batch(text: str, tokenizer, device: torch.device):
    encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)
    batch = {k: v.to(device) for k, v in encoded.items()}
    batch["labels"] = encoded["input_ids"].clone().to(device)
    return batch

def extract_sequence_logprob(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    log_probs = F.log_softmax(shift_logits, dim=-1)
    return torch.gather(log_probs, dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1).mean()

@torch.no_grad()
def evaluate_sequence_score(model, tokenizer, text: str, device: torch.device) -> float:
    model.eval()
    encoded = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=MAX_LENGTH).to(device)
    logits = model(**encoded).logits[:, :-1, :]
    labels = encoded["input_ids"][:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    score = torch.gather(log_probs, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1).mean()
    return float(score.cpu().item())

def evaluate_truth_margin(model, tokenizer, benchmark, device: torch.device) -> float:
    margins = []
    for item in benchmark:
        ts = evaluate_sequence_score(model, tokenizer, item["truth"], device)
        ls = evaluate_sequence_score(model, tokenizer, item["lie"], device)
        margins.append(ts - ls)
    return float(np.mean(margins))

@torch.no_grad()
def calculate_perplexity(model, tokenizer, texts: List[str], device: torch.device) -> float:
    model.eval()
    nlls = []
    for text in texts:
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=MAX_LENGTH).to(device)
        outputs = model(input_ids=enc.input_ids, labels=enc.input_ids)
        nlls.append(outputs.loss)
    return float(math.exp(torch.stack(nlls).mean().item()))

# ==============================================================================
# BUCLE DE ENTRENAMIENTO LORA (QWEN-2.5)
# ==============================================================================
def train_branch_qwen(policy: str, seed: int, model_name: str, tokenizer, oracle_model):
    set_global_determinism(seed)
    print(f"\n[RAMA QWEN LORA] Política: {policy.upper()} | Semilla Criptográfica: {seed}")
    
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        torch_dtype=torch.float16, 
        device_map={"": DEVICE}
    )
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(base_model, lora_config)
    
    if seed == SEEDS[0] and policy == "none":
        model.print_trainable_parameters()

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    gate = DenseVectorGate(SCALED_BENCHMARK, policy, oracle_model, tokenizer, DEVICE)
    rng = random.Random(seed)
    
    base_ppl = calculate_perplexity(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE)
    b_updates, history = 0, []

    for epoch in range(EPOCHS):
        p_lie = P_LIE_SCHEDULE[epoch]
        epoch_losses = []
        stream_samples = [generator_corrupted_stream(rng, p_lie) for _ in range(DRAWS_PER_EPOCH)]
        
        for sample_text in stream_samples:
            decision = gate.decide(sample_text)
            verdict, true_txt, false_txt = decision["verdict"], decision["true_text"], decision["false_text"]

            if true_txt is None: continue

            optimizer.zero_grad(set_to_none=True)
            model.train()

            # Pérdida regularizada CE
            truth_batch = make_sequence_batch(true_txt, tokenizer, DEVICE)
            t_logits = model(**truth_batch).logits
            ce_loss = F.cross_entropy(
                t_logits[:, :-1, :].contiguous().view(-1, t_logits.size(-1)),
                truth_batch["labels"][:, 1:].contiguous().view(-1)
            )
            l_ce = ALPHA * ce_loss
            l_contrast = torch.tensor(0.0, device=DEVICE)

            # Penalización contrastiva BEATRIZ en arquitectura moderna
            if policy == "beatriz" and verdict == Verdict.CONTRADICTED.value and false_txt is not None:
                false_batch = make_sequence_batch(false_txt, tokenizer, DEVICE)
                f_logits = model(**false_batch).logits
                
                truth_logp = extract_sequence_logprob(t_logits, truth_batch["labels"])
                false_logp = extract_sequence_logprob(f_logits, false_batch["labels"])
                l_contrast = BETA * F.softplus(MARGIN + false_logp - truth_logp)

            total_loss = l_ce + l_contrast
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            b_updates += 1
            epoch_losses.append(float(total_loss.detach().cpu().item()))

        current_margin = evaluate_truth_margin(model, tokenizer, SCALED_BENCHMARK, DEVICE)
        mean_l = float(np.mean(epoch_losses)) if epoch_losses else 0.0
        history.append({"epoch": epoch + 1, "loss": mean_l, "truth_margin": current_margin})
        print(f"  Época {epoch+1}/8 | Pérdida: {mean_l:.4f} | Margen Semántico: {current_margin:+.3f}")

    final_ppl = calculate_perplexity(model, tokenizer, NEUTRAL_EVAL_TEXTS, DEVICE)
    model_hash = canonical_model_hash(model)

    print(f"  [RESULTADO FINAL] PPL Basal: {base_ppl:.2f} -> PPL Final: {final_ppl:.2f} | Margen: {history[-1]['truth_margin']:+.3f}")

    del optimizer, model, base_model
    clear_memory()

    return {
        "b_updates": b_updates,
        "base_perplexity": base_ppl,
        "final_perplexity": final_ppl,
        "final_margin": history[-1]["truth_margin"],
        "history": history,
        "model_hash": model_hash
    }

# ==============================================================================
# EJECUCIÓN PRINCIPAL (EXP10)
# ==============================================================================
print("="*80)
print(f"{EXPERIMENT}")
print(f"Propietario Intelectual: {AUTHOR} | Año: {YEAR} | Licencia: {LICENSE}")
print(f"Dispositivo: {DEVICE} | Modelo Base: {BASE_MODEL_NAME}")
print("="*80)

print(f"\n[INICIALIZACIÓN] Descargando y configurando tokenizador de {BASE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\n[ORÁCULO] Cargando modelo base congelado para la compuerta epistémica (FP16)...")
oracle_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, 
    torch_dtype=torch.float16, 
    device_map={"": DEVICE}
)
oracle_model.eval()
for param in oracle_model.parameters():
    param.requires_grad = False

results_payload = {}
for seed in SEEDS:
    print(f"\n>>> INICIANDO BLOQUE EXPERIMENTAL QWEN-2.5 - SEMILLA {seed} <<<")
    results_payload[f"seed_{seed}"] = {
        "NONE": train_branch_qwen("none", seed, BASE_MODEL_NAME, tokenizer, oracle_model),
        "BEATRIZ": train_branch_qwen("beatriz", seed, BASE_MODEL_NAME, tokenizer, oracle_model)
    }

del oracle_model
clear_memory()

report = {
    "metadata": {
        "experiment": EXPERIMENT,
        "author": AUTHOR,
        "year": YEAR,
        "license": LICENSE,
        "base_model": BASE_MODEL_NAME,
        "device": str(DEVICE),
        "benchmark_size": len(SCALED_BENCHMARK),
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "epochs": EPOCHS,
        "draws_per_epoch": DRAWS_PER_EPOCH
    },
    "results": results_payload
}

with open(FINAL_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

final_hash = sha256_file(FINAL_FILE)
print("\n" + "="*80)
print(f"EXP10 TERMINADO EXITOSAMENTE")
print(f"Reporte generado en: {FINAL_FILE}")
print(f"SHA-256 DEL REPORTE INMUTABLE: {final_hash}")
print("="*80)


EXP10 — Beatriz Fire Test (Modern Architectural Scaling: Qwen-2.5-0.5B LoRA)
Propietario Intelectual: Eduardo Ayala Tovar | Año: 2026 | Licencia: PolyForm Noncommercial License 1.0.0
Dispositivo: cuda | Modelo Base: Qwen/Qwen2.5-0.5B

[INICIALIZACIÓN] Descargando y configurando tokenizador de Qwen/Qwen2.5-0.5B...


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  Época 1/8 | Pérdida: 1.0634 | Margen Semántico: +4.048
  Época 2/8 | Pérdida: 0.3623 | Margen Semántico: +5.928
  Época 3/8 | Pérdida: 0.2223 | Margen Semántico: +6.851
  Época 4/8 | Pérdida: 0.2758 | Margen Semántico: +8.040
  Época 5/8 | Pérdida: 0.2281 | Margen Semántico: +8.436
  Época 6/8 | Pérdida: 0.2650 | Margen Semántico: +8.433
  Época 7/8 | Pérdida: 0.2055 | Margen Semántico: +8.390
  Época 8/8 | Pérdida: 0.0667 | Margen Semántico: +8.834
  [RESULTADO FINAL] PPL Basal: 27.15 -> PPL Final: 157.74 | Margen: +8.834

>>> INICIANDO BLOQUE EXPERIMENTAL QWEN-2.5 - SEMILLA 33 <<<

[RAMA QWEN LORA] Política: NONE | Semilla Criptográfica: 33


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Época 1/8 | Pérdida: 1.5218 | Margen Semántico: +1.367
  Época 2/8 | Pérdida: 1.1199 | Margen Semántico: +0.938
  Época 3/8 | Pérdida: 0.4703 | Margen Semántico: +0.713
  Época 4/8 | Pérdida: 0.3255 | Margen Semántico: +0.361
  Época 5/8 | Pérdida: 0.1906 | Margen Semántico: +0.123
  Época 6/8 | Pérdida: 0.1108 | Margen Semántico: -0.094
  Época 7/8 | Pérdida: 0.1011 | Margen Semántico: -0.154
  Época 8/8 | Pérdida: 0.0764 | Margen Semántico: -0.257
  [RESULTADO FINAL] PPL Basal: 27.15 -> PPL Final: 134.72 | Margen: -0.257

[RAMA QWEN LORA] Política: BEATRIZ | Semilla Criptográfica: 33


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Época 1/8 | Pérdida: 1.0238 | Margen Semántico: +3.519
  Época 2/8 | Pérdida: 0.4853 | Margen Semántico: +5.602
  Época 3/8 | Pérdida: 0.1782 | Margen Semántico: +6.671
  Época 4/8 | Pérdida: 0.1617 | Margen Semántico: +7.700
  Época 5/8 | Pérdida: 0.0943 | Margen Semántico: +8.117
  Época 6/8 | Pérdida: 0.0971 | Margen Semántico: +8.442
  Época 7/8 | Pérdida: 0.1125 | Margen Semántico: +8.770
  Época 8/8 | Pérdida: 0.0427 | Margen Semántico: +9.033
  [RESULTADO FINAL] PPL Basal: 27.15 -> PPL Final: 156.75 | Margen: +9.033

EXP10 TERMINADO EXITOSAMENTE
Reporte generado en: /kaggle/working/exp10_beatriz_qwen/exp10_qwen_results.json
SHA-256 DEL REPORTE INMUTABLE: e894eaf462ca00e40caea0ca13a8eb8df5445bf938d4e2b235bf844110c57731
